# Variables, Types, and Conditionals

We have just read about why analysts write code: reproducibility, scale, auditability, and the ability to automate work that currently eats our Monday mornings. Now we are going to write some.

Our setting for this unit is the International Space Monitoring Agency. Our first task is to process telemetry from a single satellite, **Sentinel-1**. The data is small and the operations are simple, but every technique here will scale to the full constellation later.

By the end of this notebook we will be able to:

- Store and retrieve values using **variables** with appropriate data types
- Write **conditional statements** that produce different outputs based on data values
- Predict the output of a short piece of code that combines variables and conditionals

---

## 1. Storing telemetry in variables

In a spreadsheet, we put a value in a cell and refer to it by its address: A1, B3, G12. In Python, we give the value a **name** instead. That name is called a variable, and we create one with a single `=` sign. The name goes on the left, the value on the right.

Here is a telemetry snapshot from Sentinel-1:

In [ ]:
# Satellite identification
satellite_id = "SAT-001"
name = "Sentinel-1"

# Orbital parameters
orbit_altitude_km = 520.3
velocity_kms = 7.59

# Health metrics
battery_pct = 87.4
signal_strength_dbm = -68.2
temperature_c = 22.5

# Status flags
status = "active"
is_sunlit = True

Run the cell above, then run this one. Each variable now holds a value we can retrieve by name:

In [ ]:
print(name)
print(orbit_altitude_km)
print(is_sunlit)

Notice the naming style: `orbit_altitude_km`, not `OrbitAltitude` or `orbitaltitudekm`. Python convention is **snake_case** (lowercase words separated by underscores). Good variable names read like labels, so anyone reviewing the code can tell what each value represents.

---

## 2. The four fundamental types

Every value in Python has a **type** that determines what we can do with it. We would not try to multiply a name by a temperature, and Python will not let us either. There are four types we will use constantly:

- **`int`**: whole numbers like `520`
- **`float`**: decimal numbers like `520.3`
- **`str`**: text, always in quotes, like `"Sentinel-1"`
- **`bool`**: either `True` or `False`

Use the built-in `type()` function to check what type a variable holds:

In [ ]:
print(type(satellite_id))
print(type(orbit_altitude_km))
print(type(battery_pct))
print(type(is_sunlit))

Notice that `orbit_altitude_km` is a `float` because the value `520.3` has a decimal point. If we wrote `520` without the decimal, Python would store it as an `int`. The distinction matters when we need precision in calculations.

In [ ]:
print(type(520))    # int
print(type(520.0))  # float — the .0 makes it a float

---

## 3. Arithmetic with numbers

We already know these operations from spreadsheet formulas. The symbols are mostly the same: `+`, `-`, `*`, `/` all work as we would expect. Python adds a few extras:

- `//` (floor division): divides and rounds down to a whole number
- `%` (modulo): gives the remainder after division
- `**` (exponent): raises to a power

### Temperature conversion

Sentinel-1's onboard thermometer reads in Celsius, but ground control in Houston needs Fahrenheit. This is the kind of formula we might write in a spreadsheet cell. Here it is as Python:

In [ ]:
temperature_c = 22.5
temperature_f = (temperature_c * 9 / 5) + 32

print(temperature_c, "°C")
print(temperature_f, "°F")

### Altitude calculations

Now a slightly more involved example. Sentinel-1 orbits at 520.3 km above the Earth's surface. How far is that from the centre of the Earth? And how many full orbits does it complete in a day? Floor division and modulo are useful here.

In [ ]:
earth_radius_km = 6371
orbit_altitude_km = 520.3

distance_from_centre = earth_radius_km + orbit_altitude_km
print(distance_from_centre, "km from Earth's centre")

# How many full orbits in 24 hours if one orbit takes ~95 minutes?
orbit_period_min = 95
minutes_in_day = 24 * 60
full_orbits = minutes_in_day // orbit_period_min
remaining_minutes = minutes_in_day % orbit_period_min

print(full_orbits, "full orbits per day")
print(remaining_minutes, "minutes into the next orbit")

---

## 4. f-strings: building status reports

So far, `print()` has been getting the job done. But satellite operators need formatted reports, not raw numbers. An **f-string** (formatted string literal) lets us embed variables and expressions directly inside a string. Prefix the string with `f` and wrap expressions in curly braces `{}`.

In [ ]:
print(f"Satellite: {name}")
print(f"Altitude: {orbit_altitude_km} km")
print(f"Battery: {battery_pct}%")

### Format specifiers

We can control how values appear by adding a colon `:` inside the braces. A few we will use often:

- `:.2f` gives two decimal places (e.g. `87.40`)
- `:,` adds a thousands separator (e.g. `6,371`)
- `:>10` right-aligns in a 10-character field
- `:+.1f` always shows the sign (useful for signal strength, where negative values are the norm)

In [ ]:
# Format specifiers in action
print(f"Battery: {battery_pct:.2f}%")
print(f"Distance from centre: {distance_from_centre:,.1f} km")
print(f"Signal: {signal_strength_dbm:+.1f} dBm")

### Building a multi-line status report

With f-strings and format specifiers, we can produce clean, aligned output. This is the kind of report a ground station operator would actually read:

In [ ]:
report = f"""========== SATELLITE STATUS REPORT ==========
ID:              {satellite_id}
Name:            {name}
Status:          {status.upper()}
Sunlit:          {is_sunlit}
----------------------------------------------
Altitude:        {orbit_altitude_km:>10.1f} km
Velocity:        {velocity_kms:>10.2f} km/s
Battery:         {battery_pct:>10.1f} %
Signal:          {signal_strength_dbm:>+10.1f} dBm
Temperature:     {temperature_c:>10.1f} °C
=============================================="""

print(report)

---

## 5. Type conversion: when telemetry arrives as text

When we import data from a CSV in a spreadsheet, the software guesses what is a number and what is text. Sometimes it guesses wrong, and we get numbers stored as strings that silently break our formulas.

Python makes this explicit. When we read data from a file or a network stream, everything starts as a string. If we want to do arithmetic with it, we have to convert it ourselves.

In [ ]:
raw_altitude = "520.3"
print(type(raw_altitude))  # str, not float

Try to do maths with that string and Python raises a **TypeError**. This is actually a good thing: it tells us something is wrong rather than silently giving us a bad result.

In [ ]:
# This will crash — uncomment to see the error
# raw_altitude + 100

The fix is **type conversion**, converting a value from one type to another using built-in functions:

- `int("520")` converts text to an integer
- `float("520.3")` converts text to a decimal number
- `str(520.3)` converts a number to text
- `bool(1)` converts to a boolean (`True`)

In [ ]:
raw_altitude = "520.3"
altitude = float(raw_altitude)

print(altitude + 100)  # 620.3 — works now
print(type(altitude))  # float

### Parsing a raw telemetry line

In practice, satellite telemetry often arrives as a single string with fields separated by a delimiter, much like a row in a CSV file. We need to split the string apart and convert each field to the right type before we can work with it.

In [ ]:
raw_telemetry = "SAT-001,Sentinel-1,520.3,7.59,87.4,-68.2,22.5,active,True"

fields = raw_telemetry.split(",")
print(fields)
print(type(fields[2]))  # Still a string

In [ ]:
# Parse each field into the correct type
sat_id = fields[0]
sat_name = fields[1]
alt_km = float(fields[2])
vel_kms = float(fields[3])
batt_pct = float(fields[4])
sig_dbm = float(fields[5])
temp_c = float(fields[6])
sat_status = fields[7]
sunlit = fields[8] == "True"  # Convert string "True"/"False" to bool

print(f"{sat_name}: altitude={alt_km} km, battery={batt_pct}%")
print(f"Sunlit: {sunlit} (type: {type(sunlit).__name__})")

---

## 6. String methods: cleaning messy data

Anyone who has worked with real data knows this problem: extra spaces, inconsistent capitalisation, underscores where there should be hyphens. In a spreadsheet, we might use TRIM or UPPER. Python strings have their own **methods** (operations you call on the string itself) that do the same work.

The most useful ones:

- `.strip()` removes leading and trailing whitespace
- `.lower()` / `.upper()` converts case
- `.split(sep)` breaks a string into a list at each separator
- `.replace(old, new)` swaps one substring for another
- `.startswith()` / `.endswith()` checks how a string begins or ends

In [ ]:
# Messy data from a ground station
messy_status = "  Active  "
messy_id = "sat_001"
messy_name = "  SENTINEL-1  "

clean_status = messy_status.strip().lower()
clean_id = messy_id.upper().replace("_", "-")
clean_name = messy_name.strip()

print(f"Status: '{clean_status}'")
print(f"ID: '{clean_id}'")
print(f"Name: '{clean_name}'")

### Method chaining

Because each string method returns a new string, we can **chain** multiple calls together, left to right. This is the same idea as nesting functions in a spreadsheet formula, but easier to read.

In [ ]:
raw_input = "  STATUS: Active  "

# Chain: strip whitespace → convert to lowercase → remove the "status: " prefix
cleaned = raw_input.strip().lower().replace("status: ", "")
print(cleaned)  # "active"

In [ ]:
# Checking string content
filename = "telemetry_2026-03-21.csv"

print(filename.startswith("telemetry"))  # True
print(filename.endswith(".csv"))          # True
print(filename.find("2026"))              # 10 — position where "2026" starts

---

## 7. Booleans and comparisons

A **boolean** is either `True` or `False`. If we have ever written an IF formula in a spreadsheet, the condition inside it (`A1 > 50`, `B2 = "active"`) evaluates to true or false. Python works the same way, just with slightly different syntax.

### Comparison operators

| Operator | Meaning | Example | Result |
|----------|---------|---------|--------|
| `<` | Less than | `battery_pct < 50` | `False` |
| `>` | Greater than | `battery_pct > 50` | `True` |
| `==` | Equal to | `status == "active"` | `True` |
| `!=` | Not equal to | `status != "active"` | `False` |
| `<=` | Less than or equal | `battery_pct <= 87.4` | `True` |
| `>=` | Greater than or equal | `battery_pct >= 90` | `False` |

One thing to watch: **equality is `==` (double equals), not `=` (single equals)**. Single `=` is for assigning a variable. This trips everyone up at first.

In [ ]:
print(battery_pct > 50)        # True
print(battery_pct < 20)        # False
print(status == "active")      # True
print(orbit_altitude_km >= 500)  # True

### Combining conditions with `and`, `or`, `not`

In a spreadsheet we might write `=AND(A1>50, B1="active")`. Python uses the words `and`, `or`, and `not` directly:

- `and` requires both conditions to be true
- `or` requires at least one condition to be true
- `not` flips true to false and vice versa

In [ ]:
# Is the satellite in a good state?
good_battery = battery_pct > 50
good_signal = signal_strength_dbm > -80
is_active = status == "active"

print(f"Good battery: {good_battery}")
print(f"Good signal: {good_signal}")
print(f"Active: {is_active}")

# All conditions must be true
all_good = good_battery and good_signal and is_active
print(f"All systems nominal: {all_good}")

# At least one warning sign
any_concern = not good_battery or not good_signal
print(f"Any concerns: {any_concern}")

---

## 8. Making decisions with conditionals

Comparisons tell us whether something is true. Conditionals let us **act** on that information. This is the Python equivalent of an IF/ELSE formula, but we can chain as many conditions as we need.

```python
if condition:
    # runs when condition is True
elif another_condition:
    # runs when the first was False and this one is True
else:
    # runs when all conditions above were False
```

The indented block under each branch is critical. Python uses **indentation** (4 spaces) to define which code belongs to which branch. No braces, no `END IF`, just the indentation.

### Classifying battery level

Sentinel-1's battery is at 87.4%. Is that good? The ground team uses four categories. Here is how to express that classification in code:

In [ ]:
battery_pct = 87.4

if battery_pct < 20:
    battery_status = "CRITICAL"
elif battery_pct < 40:
    battery_status = "LOW"
elif battery_pct < 80:
    battery_status = "GOOD"
else:
    battery_status = "EXCELLENT"

print(f"Battery: {battery_pct}% — {battery_status}")

Python checks the conditions top to bottom. As soon as one is `True`, its block runs and the rest are skipped. Since 87.4 is not less than 20, 40, or 80, execution falls through to the `else` block.

### Classifying signal strength

Signal strength is measured in dBm, where values closer to 0 are stronger. A reading of -68.2 dBm needs a different classification scale:

In [ ]:
signal_strength_dbm = -68.2

# Signal strength in dBm: closer to 0 is stronger
if signal_strength_dbm > -50:
    signal_status = "EXCELLENT"
elif signal_strength_dbm > -70:
    signal_status = "GOOD"
elif signal_strength_dbm > -85:
    signal_status = "WEAK"
else:
    signal_status = "CRITICAL"

print(f"Signal: {signal_strength_dbm} dBm — {signal_status}")

### Combining conditions for overall status

In practice, a satellite's health depends on several readings at once. We can use `and`, `or`, and `not` inside conditionals to express this. The logic below mirrors the kind of multi-condition IF we might build in a spreadsheet, except it reads top to bottom instead of nesting deeper and deeper.

In [ ]:
battery_pct = 87.4
signal_strength_dbm = -68.2
temperature_c = 22.5
status = "active"

if status != "active":
    overall = "OFFLINE"
elif battery_pct < 20 or signal_strength_dbm < -85:
    overall = "CRITICAL"
elif battery_pct < 40 or signal_strength_dbm < -70 or temperature_c > 45:
    overall = "ATTENTION NEEDED"
else:
    overall = "HEALTHY"

print(f"Satellite {name}: {overall}")

### Predict the output

Before running the next cell, read the code and work out what it will print. This is a useful habit: predicting output forces us to trace through the logic the way Python does.

In [ ]:
altitude = 520.3
speed = 7.59
is_operational = True

if altitude > 600 and is_operational:
    print("High orbit, operational")
elif altitude > 400 or speed > 8:
    print("Standard parameters")
else:
    print("Low orbit warning")

The answer is `"Standard parameters"`. The first condition fails because `altitude > 600` is `False` (520.3 is not greater than 600). The second condition succeeds because `altitude > 400` is `True`, and with `or`, only one side needs to be `True`.

---

## 9. Exercises

Time to write some code. Each exercise uses the techniques from this notebook. If you get stuck, scroll back to the relevant section.

### Exercise 1: New satellite telemetry

Create variables for a different satellite with the following data:

- ID: `"SAT-007"`
- Name: `"Aurora-3"`
- Altitude: `410` km
- Battery: `63.1`%
- Signal strength: `-74.8` dBm
- Status: `"active"`
- Sunlit: `False`

Print the value and type of each variable.

In [ ]:
# Your code here


### Exercise 2: Temperature conversion

You receive three temperature readings as strings: `"18.7"`, `"-3.2"`, `"41.0"`. Convert each to a float, then compute the Fahrenheit equivalent using the formula `F = C * 9/5 + 32`. Print each result formatted to one decimal place.

In [ ]:
# Your code here


### Exercise 3: Parse a semicolon-separated telemetry string

The following telemetry string uses semicolons as separators:

```
"SAT-012;Horizon-5;485.7;7.62;45.2;-81.3;38.9;active;True"
```

Split the string, convert numeric fields to the appropriate types, and print a formatted status report using f-strings.

In [ ]:
# Your code here


### Exercise 4: Clean messy status strings

You have received the following status strings from different ground stations. Clean each one so it is lowercase, stripped of whitespace, and has underscores replaced with spaces:

```python
s1 = "  ACTIVE  "
s2 = "  Partially_Operational "
s3 = "DECOMMISSIONED  "
```

Print the cleaned version of each.

In [ ]:
# Your code here


### Exercise 5: Satellite health classifier

Write conditional logic to classify a satellite's health as `"healthy"`, `"attention needed"`, or `"critical"`. Use the following rules:

- **Critical** if any of: battery below 20%, signal below -85 dBm, or temperature above 50 °C
- **Attention needed** if any of: battery below 40%, signal below -70 dBm, or temperature above 40 °C
- **Healthy** otherwise

Test with these values: `battery_pct = 35.0`, `signal_strength_dbm = -72.5`, `temperature_c = 28.0`.

In [ ]:
# Your code here


---

## Summary

We have written our first Python code. We stored satellite telemetry in named variables, performed calculations, built formatted reports, cleaned messy strings, and used conditionals to classify readings. All operations we have done before in spreadsheets, now expressed as readable, repeatable instructions.

The key ideas:

- **Variables** give meaningful names to values. `battery_pct = 87.4` is clearer than cell G14.
- **Types** (`int`, `float`, `str`, `bool`) determine what operations a value supports, and Python will tell us when we get it wrong.
- **f-strings** let us build formatted output with embedded expressions.
- **Type conversion** is explicit: we decide when a string becomes a number, rather than letting the software guess.
- **String methods** (`.strip()`, `.lower()`, `.split()`, `.replace()`) handle the messy-data problems we already know from spreadsheets.
- **Conditionals** (`if`/`elif`/`else`) let us branch logic based on data values, like IF formulas but more readable.

Right now, everything we have done applies to a single satellite. In the next notebook, we will work with **collections** — lists, dictionaries, and loops — to handle telemetry from the full constellation.